In [6]:
import os, csv, time, random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

# ── Device ────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device  : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

# ── Paths ─────────────────────────────────────────────────────────
TRAIN_DIR = Path('../Dataset/train/train')
VALID_DIR = Path('../Dataset/valid/valid')

# ── Hyperparams ───────────────────────────────────────────────────
IMG_SIZE    = 128
BATCH_SIZE  = 64
EPOCHS      = 100
LR          = 2e-4
T_TOTAL     = 1000       # total diffusion steps
T_NOISE     = 400        # partial diffusion noise level at test time
BETA_START  = 1e-4
BETA_END    = 0.02
SAVE_DIR    = Path('checkpoints/anodddpm')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
LOG_CSV     = SAVE_DIR / 'anodddpm_training_log.csv'

Device  : cuda
GPU     : NVIDIA GeForce RTX 4090
VRAM    : 24.0 GB


In [7]:
# ── Dataset ───────────────────────────────────────────────────────
def get_image_paths(folder):
    exts = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
    return sorted([p for p in Path(folder).rglob('*')
                   if p.suffix.lower() in exts
                   and '__MACOSX' not in str(p)])

class IXIDataset(Dataset):
    def __init__(self, folder, img_size=128):
        self.paths = get_image_paths(folder)
        self.tf = T.Compose([
            T.Grayscale(1),
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize([0.5], [0.5]),
        ])
        print(f'  {Path(folder).name}: {len(self.paths)} images')

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        return self.tf(Image.open(self.paths[idx]).convert('RGB'))

print('Loading datasets...')
train_ds = IXIDataset(TRAIN_DIR, IMG_SIZE)
val_ds   = IXIDataset(VALID_DIR, IMG_SIZE)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=0,
                          pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0)


# ── Simplex noise ─────────────────────────────────────────────────
def generate_simplex_noise(shape, device, octaves=6):
    """
    Approximates simplex noise via summed smoothed Gaussian noise.
    Produces spatially correlated noise — structures closer to
    tumour texture than pure Gaussian noise.
    """
    B, C, H, W = shape
    noise = torch.zeros(shape, device=device)
    freq  = 1.0
    amp   = 1.0
    total_amp = 0.0
    for _ in range(octaves):
        size_h = max(1, int(H / freq))
        size_w = max(1, int(W / freq))
        layer  = torch.randn(B, C, size_h, size_w, device=device)
        layer  = F.interpolate(layer, size=(H, W),
                               mode='bilinear', align_corners=False)
        noise     += amp * layer
        total_amp += amp
        freq      *= 2.0
        amp       *= 0.5
    return noise / total_amp


# ── Diffusion schedule ────────────────────────────────────────────
class DiffusionSchedule:
    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02, device='cpu'):
        self.T      = T
        self.device = device
        betas       = torch.linspace(beta_start, beta_end, T, device=device)
        alphas      = 1.0 - betas
        alpha_bar   = torch.cumprod(alphas, dim=0)

        self.betas      = betas
        self.alphas     = alphas
        self.alpha_bar  = alpha_bar
        self.sqrt_ab    = alpha_bar.sqrt()
        self.sqrt_1mab  = (1 - alpha_bar).sqrt()

    def q_sample(self, x0, t, noise=None, use_simplex=True):
        """
        Forward process: add noise to x0 at timestep t.
        AnoDDPM key: use simplex noise instead of Gaussian.
        """
        if noise is None:
            noise = generate_simplex_noise(x0.shape, x0.device) \
                    if use_simplex else torch.randn_like(x0)

        sqrt_ab   = self.sqrt_ab[t].view(-1, 1, 1, 1)
        sqrt_1mab = self.sqrt_1mab[t].view(-1, 1, 1, 1)
        return sqrt_ab * x0 + sqrt_1mab * noise, noise

    @torch.no_grad()
    def p_sample(self, model, xt, t_scalar):
        """Single denoising step."""
        t_tensor = torch.full((xt.size(0),), t_scalar,
                              device=self.device, dtype=torch.long)
        beta_t      = self.betas[t_scalar]
        alpha_t     = self.alphas[t_scalar]
        alpha_bar_t = self.alpha_bar[t_scalar]

        eps_pred = model(xt, t_tensor)
        coef     = beta_t / (1 - alpha_bar_t).sqrt()
        mean     = (1 / alpha_t.sqrt()) * (xt - coef * eps_pred)

        if t_scalar > 0:
            noise = generate_simplex_noise(xt.shape, self.device)
            std   = beta_t.sqrt()
            return mean + std * noise
        return mean

    @torch.no_grad()
    def reconstruct(self, model, x0, t_noise=400):
        """
        AnoDDPM inference:
        1. Add simplex noise up to t_noise
        2. Denoise all the way back to t=0
        Healthy tissue → reconstructed well → small residual
        Anomaly      → can't reconstruct → large residual
        """
        model.eval()
        t_tensor = torch.full((x0.size(0),), t_noise,
                              device=self.device, dtype=torch.long)
        xt, _ = self.q_sample(x0, t_tensor, use_simplex=True)

        for t in reversed(range(t_noise)):
            xt = self.p_sample(model, xt, t)
        return xt


schedule = DiffusionSchedule(T_TOTAL, BETA_START, BETA_END, DEVICE)
print('Diffusion schedule ready ✓')
print(f'  T_total={T_TOTAL}  T_noise={T_NOISE}  beta [{BETA_START}→{BETA_END}]')

Loading datasets...
  train: 25447 images
  valid: 2828 images
Train: 25447 | Val: 2828
Diffusion schedule ready ✓
  T_total=1000  T_noise=400  beta [0.0001→0.02]


In [9]:
# ── Sinusoidal time embedding ─────────────────────────────────────
class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.proj = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.SiLU(),
            nn.Linear(dim * 4, dim),
        )

    def forward(self, t):
        half = self.dim // 2
        freq = torch.exp(
            -torch.arange(half, device=t.device).float()
            * (torch.log(torch.tensor(10000.0)) / (half - 1))
        )
        args = t[:, None].float() * freq[None]
        emb  = torch.cat([args.sin(), args.cos()], dim=-1)
        return self.proj(emb)


# ── Residual block with time conditioning ────────────────────────
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.norm1  = nn.GroupNorm(min(8, in_ch),  in_ch)
        self.conv1  = nn.Conv2d(in_ch,  out_ch, 3, padding=1)
        self.norm2  = nn.GroupNorm(min(8, out_ch), out_ch)
        self.conv2  = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.t_proj = nn.Linear(time_dim, out_ch)
        self.skip   = nn.Conv2d(in_ch, out_ch, 1) \
                      if in_ch != out_ch else nn.Identity()
        self.act    = nn.SiLU()

    def forward(self, x, t_emb):
        h  = self.act(self.norm1(x))
        h  = self.conv1(h)
        h  = h + self.t_proj(self.act(t_emb))[:, :, None, None]
        h  = self.act(self.norm2(h))
        h  = self.conv2(h)
        return h + self.skip(x)


# ── Attention block ───────────────────────────────────────────────
class AttnBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.norm = nn.GroupNorm(min(8, ch), ch)
        self.qkv  = nn.Conv2d(ch, ch * 3, 1)
        self.proj = nn.Conv2d(ch, ch, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        h   = self.norm(x)
        qkv = self.qkv(h).reshape(B, 3, C, H*W)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
        scale = C ** -0.5
        attn  = torch.softmax((q.transpose(-1,-2) @ k) * scale, dim=-1)
        out   = (attn @ v.transpose(-1,-2)).transpose(-1,-2)
        out   = out.reshape(B, C, H, W)
        return x + self.proj(out)


# ── Down / Up blocks ──────────────────────────────────────────────
class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, use_attn=False):
        super().__init__()
        self.res  = ResBlock(in_ch, out_ch, time_dim)
        self.attn = AttnBlock(out_ch) if use_attn else nn.Identity()
        self.down = nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1)

    def forward(self, x, t):
        x = self.res(x, t)
        x = self.attn(x)
        return self.down(x), x   # (downsampled, skip)


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, time_dim, use_attn=False):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, in_ch, 2, stride=2)
        self.res  = ResBlock(in_ch + skip_ch, out_ch, time_dim)
        self.attn = AttnBlock(out_ch) if use_attn else nn.Identity()

    def forward(self, x, skip, t):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        x = self.res(x, t)
        return self.attn(x)


# ── U-Net ─────────────────────────────────────────────────────────
class UNet(nn.Module):
    """
    Lightweight U-Net for 128×128 single-channel brain MRI.
    ch = [64, 128, 256, 512]
    Attention at 16×16 and 8×8 resolution.
    """
    def __init__(self, in_ch=1, base_ch=64, time_dim=256):
        super().__init__()
        ch = [base_ch, base_ch*2, base_ch*4, base_ch*8]

        self.t_emb  = TimeEmbedding(time_dim)
        self.init   = nn.Conv2d(in_ch, ch[0], 3, padding=1)

        # encoder
        self.down1  = DownBlock(ch[0], ch[1], time_dim, use_attn=False)
        self.down2  = DownBlock(ch[1], ch[2], time_dim, use_attn=False)
        self.down3  = DownBlock(ch[2], ch[3], time_dim, use_attn=True)

        # bottleneck
        self.mid1   = ResBlock(ch[3], ch[3], time_dim)
        self.mid_a  = AttnBlock(ch[3])
        self.mid2   = ResBlock(ch[3], ch[3], time_dim)

        # decoder
        self.up3    = UpBlock(ch[3], ch[3], ch[2], time_dim, use_attn=True)
        self.up2    = UpBlock(ch[2], ch[2], ch[1], time_dim, use_attn=False)
        self.up1    = UpBlock(ch[1], ch[1], ch[0], time_dim, use_attn=False)

        self.final  = nn.Sequential(
            nn.GroupNorm(min(8, ch[0]), ch[0]),
            nn.SiLU(),
            nn.Conv2d(ch[0], in_ch, 1),
        )

    def forward(self, x, t):
        t_emb = self.t_emb(t)
        x     = self.init(x)

        x, s1 = self.down1(x, t_emb)
        x, s2 = self.down2(x, t_emb)
        x, s3 = self.down3(x, t_emb)

        x = self.mid1(x, t_emb)
        x = self.mid_a(x)
        x = self.mid2(x, t_emb)

        x = self.up3(x, s3, t_emb)
        x = self.up2(x, s2, t_emb)
        x = self.up1(x, s1, t_emb)

        return self.final(x)


# ── Build & sanity check ──────────────────────────────────────────
unet   = UNet(in_ch=1, base_ch=64, time_dim=256).to(DEVICE)
params = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print(f'U-Net parameters: {params/1e6:.2f}M')

with torch.no_grad():
    _x = torch.zeros(2, 1, IMG_SIZE, IMG_SIZE).to(DEVICE)
    _t = torch.randint(0, T_TOTAL, (2,)).to(DEVICE)
    _o = unet(_x, _t)
    print(f'Input : {_x.shape}')
    print(f'Output: {_o.shape}')
    print('Sanity check passed ✓')

U-Net parameters: 26.46M
Input : torch.Size([2, 1, 128, 128])
Output: torch.Size([2, 1, 128, 128])
Sanity check passed ✓


U-Net parameters: 26.46M
Input : torch.Size([2, 1, 128, 128])
Output: torch.Size([2, 1, 128, 128])
Sanity check passed ✓


In [11]:
optimizer = torch.optim.AdamW(unet.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=EPOCHS)

# ── CSV log ───────────────────────────────────────────────────────
csv_fields = ['epoch', 'loss', 'epoch_time_sec', 'lr',
              'vram_used_mb', 'vram_total_mb', 'vram_pct',
              'best_loss', 'is_best']

if not LOG_CSV.exists():
    with open(LOG_CSV, 'w', newline='') as f:
        csv.DictWriter(f, fieldnames=csv_fields).writeheader()
    print(f'New log: {LOG_CSV}')
else:
    print(f'Appending to: {LOG_CSV}')

# ── Resume ────────────────────────────────────────────────────────
RESUME      = SAVE_DIR / 'anodddpm_latest.pt'
start_epoch = 1
best_loss   = float('inf')

if RESUME.exists():
    ckpt = torch.load(RESUME, map_location=DEVICE)
    unet.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optim'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch = ckpt['epoch'] + 1
    best_loss   = ckpt['best_loss']
    print(f'Resumed from epoch {ckpt["epoch"]}, loss={best_loss:.6f}')

# ── Training loop ─────────────────────────────────────────────────
for epoch in range(start_epoch, EPOCHS + 1):
    unet.train()
    total_loss = 0.0
    t0 = time.time()

    for x0 in tqdm(train_loader,
                   desc=f'Epoch {epoch:3d}/{EPOCHS}',
                   leave=False):
        x0 = x0.to(DEVICE)
        B  = x0.size(0)

        # random timestep per image
        t = torch.randint(0, T_TOTAL, (B,), device=DEVICE).long()

        # add simplex noise
        noisy, noise_target = schedule.q_sample(x0, t, use_simplex=True)

        # predict noise
        noise_pred = unet(noisy, t)

        # simple MSE on noise prediction (standard DDPM loss)
        loss = F.mse_loss(noise_pred, noise_target)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()
    epoch_time = time.time() - t0
    avg_loss   = total_loss / len(train_loader)
    current_lr = optimizer.param_groups[0]['lr']
    is_best    = avg_loss < best_loss

    # VRAM
    if DEVICE == 'cuda':
        vram_used  = torch.cuda.memory_allocated(0) / 1024**2
        vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**2
        vram_pct   = vram_used / vram_total * 100
        torch.cuda.reset_peak_memory_stats(0)
    else:
        vram_used = vram_total = vram_pct = 0.0

    print(f'Epoch {epoch:3d}/{EPOCHS} | '
          f'Loss={avg_loss:.6f}  '
          f'LR={current_lr:.2e}  '
          f'VRAM={vram_used:.0f}MB ({vram_pct:.1f}%)  '
          f'Time={epoch_time:.1f}s'
          + ('  ✓ best' if is_best else ''))

    # CSV
    with open(LOG_CSV, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=csv_fields)
        writer.writerow({
            'epoch':          epoch,
            'loss':           round(avg_loss,   6),
            'epoch_time_sec': round(epoch_time, 2),
            'lr':             round(current_lr, 8),
            'vram_used_mb':   round(vram_used,  1),
            'vram_total_mb':  round(vram_total, 1),
            'vram_pct':       round(vram_pct,   2),
            'best_loss':      round(best_loss if not is_best
                                    else avg_loss, 6),
            'is_best':        is_best,
        })
        f.flush()
        os.fsync(f.fileno())

    # Rolling checkpoint
    torch.save({
        'epoch':     epoch,
        'model':     unet.state_dict(),
        'optim':     optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'best_loss': best_loss,
    }, SAVE_DIR / 'anodddpm_latest.pt')

    # Best checkpoint
    if is_best:
        best_loss = avg_loss
        torch.save({
            'epoch': epoch,
            'model': unet.state_dict(),
            'loss':  best_loss,
        }, SAVE_DIR / 'anodddpm_best.pt')

print(f'\nAnoDDPM training complete. Best loss: {best_loss:.6f}')

New log: checkpoints\anodddpm\anodddpm_training_log.csv


Epoch   1/100 | Loss=0.021179  LR=2.00e-04  VRAM=451MB (1.8%)  Time=160.9s  ✓ best


Epoch   2/100 | Loss=0.011051  LR=2.00e-04  VRAM=451MB (1.8%)  Time=154.9s  ✓ best


Epoch   3/100 | Loss=0.009955  LR=2.00e-04  VRAM=452MB (1.8%)  Time=154.8s  ✓ best


Epoch   4/100 | Loss=0.009265  LR=1.99e-04  VRAM=452MB (1.8%)  Time=154.9s  ✓ best


...

Epoch  62/100 | Loss=0.006626  LR=6.32e-05  VRAM=452MB (1.8%)  Time=154.7s  ✓ best


Epoch  63/100 | Loss=0.006761  LR=6.03e-05  VRAM=451MB (1.8%)  Time=154.7s


Epoch  64/100 | Loss=0.006651  LR=5.74e-05  VRAM=451MB (1.8%)  Time=154.8s


Epoch  65/100 | Loss=0.006534  LR=5.46e-05  VRAM=451MB (1.8%)  Time=155.1s  ✓ best


KeyboardInterrupt: 

In [14]:
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              roc_curve, classification_report)

# ── Load best checkpoint ──────────────────────────────────────────
ckpt = torch.load(SAVE_DIR / 'anodddpm_best.pt', map_location=DEVICE)
unet.load_state_dict(ckpt['model'])
unet.eval()
print(f'Loaded best model — epoch {ckpt["epoch"]}, loss={ckpt["loss"]:.6f}')

def denorm(t):
    return (t * 0.5 + 0.5).clamp(0, 1).cpu()

def brain_mask(img_tensor):
    return (img_tensor > -0.9).float()

# ── Synthetic tumor injector (same as Tri-VAE & GANomaly) ─────────
def inject_tumor(img_tensor, n_blobs=3,
                 intensity_range=(0.3, 0.8),
                 radius_range=(8, 20)):
    img   = img_tensor.clone()
    _, H, W = img.shape
    mask  = torch.zeros(H, W)
    brain = (img[0] > -0.8)
    rows  = brain.any(dim=1).nonzero(as_tuple=True)[0]
    cols  = brain.any(dim=0).nonzero(as_tuple=True)[0]
    if len(rows) < 10 or len(cols) < 10:
        return img, mask
    r_min, r_max = rows[0].item(), rows[-1].item()
    c_min, c_max = cols[0].item(), cols[-1].item()
    for _ in range(n_blobs):
        cy        = random.randint(r_min + 5, r_max - 5)
        cx        = random.randint(c_min + 5, c_max - 5)
        radius    = random.randint(*radius_range)
        intensity = random.uniform(*intensity_range)
        ys  = torch.arange(H).float()
        xs  = torch.arange(W).float()
        yy, xx = torch.meshgrid(ys, xs, indexing='ij')
        dist = ((yy - cy)**2 + (xx - cx)**2).sqrt()
        blob = torch.exp(-dist**2 / (2 * (radius/2)**2))
        blob = (blob / blob.max()) * intensity
        bf   = brain.float()
        img[0] += blob * bf
        mask   += (blob > 0.1).float() * bf
    return img.clamp(-1, 1), (mask > 0).float()

class SyntheticTumorDataset(Dataset):
    def __init__(self, val_folder, img_size=128, seed=42):
        random.seed(seed)
        torch.manual_seed(seed)
        self.tf = T.Compose([
            T.Grayscale(1),
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize([0.5], [0.5]),
        ])
        paths = get_image_paths(Path(val_folder))
        random.shuffle(paths)
        mid = len(paths) // 2
        self.samples = [(p, 0) for p in paths[:mid]] + \
                       [(p, 1) for p in paths[mid:]]
        print(f'  Normal: {mid} | Synthetic: {len(paths)-mid}')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img  = self.tf(Image.open(path).convert('RGB'))
        mask = torch.zeros(img.shape[1], img.shape[2])
        if label == 1:
            img, mask = inject_tumor(img)
        return img, label, mask

syn_ds     = SyntheticTumorDataset(VALID_DIR, IMG_SIZE)
syn_loader = DataLoader(syn_ds, batch_size=64,   # small batch — inference is slow
                        shuffle=False, num_workers=0)

# ── Score all images ──────────────────────────────────────────────
# AnoDDPM anomaly score = masked mean pixel residual
# (same metric as Tri-VAE for fair comparison)

# ── Score all images ──────────────────────────────────────────────
all_scores, all_true = [], []

with torch.no_grad():
    for imgs, labels, _ in tqdm(syn_loader, desc='Scoring (partial diffusion)'):
        imgs  = imgs.to(DEVICE)
        recon = schedule.reconstruct(unet, imgs, t_noise=T_NOISE)

        residual = (imgs - recon).abs()
        mask     = brain_mask(imgs)
        scores   = (residual * mask).sum(dim=[1,2,3]) / \
                    mask.sum(dim=[1,2,3]).clamp(min=1)

        all_scores.extend(scores.cpu().numpy())
        all_true.extend(labels.numpy())

all_scores = np.array(all_scores)
all_true   = np.array(all_true)

auroc = roc_auc_score(all_true, all_scores)
auprc = average_precision_score(all_true, all_scores)
fpr, tpr, thresholds = roc_curve(all_true, all_scores)
best_idx    = np.argmax(tpr - fpr)
best_thresh = thresholds[best_idx]
preds       = (all_scores >= best_thresh).astype(int)

print(f'\nAnoDDPM — Synthetic Tumor Evaluation')
print(f'──────────────────────────────────────')
print(f'AUROC      : {auroc:.4f}')
print(f'AUPRC      : {auprc:.4f}')
print(f'Threshold  : {best_thresh:.4f}  (Youden J)')
print(f'\n{classification_report(all_true, preds, target_names=["Normal","Synthetic"])}')

# ── Pre-compute reconstructions for visualisation ─────────────────
# batch all 8 vis images at once instead of one-by-one
all_imgs   = torch.stack([s[0] for s in syn_ds])
all_labels = torch.tensor([s[1] for s in syn_ds])

normal_idx = (all_labels == 0).nonzero(as_tuple=True)[0][:4]
tumor_idx  = (all_labels == 1).nonzero(as_tuple=True)[0][:4]
vis_idx    = list(normal_idx) + list(tumor_idx)

vis_imgs = all_imgs[vis_idx].to(DEVICE)          # (8, 1, 128, 128)
with torch.no_grad():
    vis_recon = schedule.reconstruct(unet, vis_imgs, t_noise=T_NOISE)
    vis_res   = (vis_imgs - vis_recon).abs()
    vis_mask  = brain_mask(vis_imgs)
    vis_scores = (vis_res * vis_mask).sum(dim=[1,2,3]) / \
                  vis_mask.sum(dim=[1,2,3]).clamp(min=1)

fig, axes = plt.subplots(4, 8, figsize=(18, 9))

for col in range(8):
    idx    = vis_idx[col]
    border = 'steelblue' if all_labels[idx] == 0 else 'crimson'
    label  = 'Normal' if all_labels[idx] == 0 else 'Synthetic'
    title  = f'{label}\n(s={vis_scores[col]:.3f})'
    gt_mask = syn_ds[idx.item()][2]

    for row, (data, cmap) in enumerate(zip(
            [denorm(all_imgs[idx]),
             denorm(vis_recon[col].cpu()),
             vis_res[col].squeeze().cpu(),
             gt_mask],
            ['gray', 'gray', 'hot', 'hot'])):
        ax = axes[row, col]
        ax.imshow(data.squeeze(), cmap=cmap)
        ax.axis('off')
        if row == 0:
            ax.set_title(title, color=border, fontsize=8)
        for spine in ax.spines.values():
            spine.set_edgecolor(border)
            spine.set_linewidth(2)
            spine.set_visible(True)

for row, lbl in enumerate(['Input', 'Reconstruction', 'Residual', 'GT mask']):
    axes[row, 0].set_ylabel(lbl, fontsize=9)

plt.suptitle(f'AnoDDPM — t_noise={T_NOISE}  |  blue=Normal  red=Synthetic', fontsize=11)
plt.tight_layout()
plt.show()

# ── ROC ───────────────────────────────────────────────────────────
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, lw=2, color='#534AB7',
         label=f'AnoDDPM  AUROC={auroc:.3f}')
plt.plot([0,1],[0,1],'--', color='gray', lw=1)
plt.scatter(fpr[best_idx], tpr[best_idx],
            color='crimson', zorder=5,
            label=f'Threshold={best_thresh:.3f}')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'ROC Curve — AnoDDPM  (t_noise={T_NOISE})')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ── Score distribution ────────────────────────────────────────────
plt.figure(figsize=(7, 4))
plt.hist(all_scores[all_true==0], bins=60, alpha=0.6,
         color='steelblue', label='Normal')
plt.hist(all_scores[all_true==1], bins=60, alpha=0.6,
         color='crimson',   label='Synthetic tumor')
plt.axvline(best_thresh, color='black', linestyle='--',
            label=f'Threshold={best_thresh:.3f}')
plt.xlabel('Anomaly score (masked mean residual)')
plt.ylabel('Count')
plt.title('AnoDDPM score distribution')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

C:\Users\Dell\AppData\Local\Temp\ipykernel_14796\199261948.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(SAVE_DIR / 'anodddpm_best.pt', map_location=

Loaded best model — epoch 65, loss=0.006534
  Normal: 1414 | Synthetic: 1414


Scoring (partial diffusion): 100%|██████████| 45/45 [34:54<00:00, 46.54s/it]



AnoDDPM — Synthetic Tumor Evaluation
──────────────────────────────────────
AUROC      : 0.6866
AUPRC      : 0.6297
Threshold  : 0.2008  (Youden J)

              precision    recall  f1-score   support

      Normal       0.67      0.63      0.65      1414
   Synthetic       0.65      0.68      0.67      1414

    accuracy                           0.66      2828
   macro avg       0.66      0.66      0.66      2828
weighted avg       0.66      0.66      0.66      2828



IndexError: too many indices for tensor of dimension 4